In [ ]:
# LIQUIDITY WATCH | NB_01_Liquidity_Producer  (une seule cellule, aucun lakehouse nécessaire)
# Photographie toutes les 5 secondes les carnets d'ordres publics de Binance, Kraken et Coinbase,
# calcule les mesures de liquidité de chaque marché et les envoie à l'Eventstream ES_LiquidityWatch.

CONNECTION_STRING = "COLLEZ_ICI_LA_CHAINE_DE_CONNEXION_CLE_PRIMAIRE"
DURATION_MIN = 45                      # durée d'exécution, en minutes
TICK_SEC = 5                           # une photo des carnets toutes les 5 secondes
STRESS_TEST = False                    # True à l'étape 11 : rejoue un épisode de stress balisé
STRESS_START_MIN, STRESS_LEN_MIN = 5, 10   # le stress commence 5 min après le lancement et dure 10 min

import subprocess, sys                 # installation sans redémarrer la session
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "azure-eventhub"])

import json, math, time, datetime as dt
import requests
from concurrent.futures import ThreadPoolExecutor
from azure.eventhub import EventHubProducerClient, EventData

fr = lambda n: f"{n:,}".replace(",", " ")   # 6 480 plutôt que 6,480
BN = "https://data-api.binance.vision/api/v3/depth?limit=1000&symbol="
KR = "https://api.kraken.com/0/public/Depth?count=500&pair="
CB = "https://api.exchange.coinbase.com/products/{}/book?level=2"
FEEDS = [  # (plateforme, marché, classe d'actif, adresse publique, sans clé)
    ("BINANCE",  "BTC-USD",  "crypto",     BN + "BTCUSDT"),
    ("BINANCE",  "ETH-USD",  "crypto",     BN + "ETHUSDT"),
    ("BINANCE",  "SOL-USD",  "crypto",     BN + "SOLUSDT"),
    ("KRAKEN",   "BTC-USD",  "crypto",     KR + "XBTUSD"),
    ("KRAKEN",   "ETH-USD",  "crypto",     KR + "ETHUSD"),
    ("KRAKEN",   "SOL-USD",  "crypto",     KR + "SOLUSD"),
    ("KRAKEN",   "USDT-USD", "stablecoin", KR + "USDTUSD"),
    ("KRAKEN",   "USDC-USD", "stablecoin", KR + "USDCUSD"),
    ("COINBASE", "BTC-USD",  "crypto",     CB.format("BTC-USD")),
    ("COINBASE", "ETH-USD",  "crypto",     CB.format("ETH-USD")),
    ("COINBASE", "SOL-USD",  "crypto",     CB.format("SOL-USD")),
    ("COINBASE", "USDT-USD", "stablecoin", CB.format("USDT-USD")),
]
session = requests.Session()
session.headers["User-Agent"] = "LiquidityWatch/1.0 (Microsoft Fabric)"


def fetch(feed):
    """Télécharge un carnet d'ordres et le remet au même format quelle que soit la plateforme."""
    venue, symbol, klass, url = feed
    t0 = time.perf_counter()
    js = session.get(url, timeout=4).json()
    latency = (time.perf_counter() - t0) * 1000
    if venue == "KRAKEN":
        if js.get("error"):
            raise RuntimeError(js["error"])
        js = next(iter(js["result"].values()))      # la clé change selon la paire (XXBTZUSD, USDTZUSD...)
    bids = sorted(((float(l[0]), float(l[1])) for l in js["bids"]), reverse=True)
    asks = sorted((float(l[0]), float(l[1])) for l in js["asks"])
    return bids, asks, latency


def liquidity(bids, asks):
    """Les mesures d'un salle des marchés : fourchette, prix pondéré, profondeur, déséquilibre."""
    bb, ba = bids[0][0], asks[0][0]
    mid = (bb + ba) / 2
    qb, qa = bids[0][1], asks[0][1]

    def depth(side, bps, below):                     # dollars disponibles à moins de N points de base du milieu
        lim = mid * (1 - bps / 1e4) if below else mid * (1 + bps / 1e4)
        return sum(p * q for p, q in side if (p >= lim if below else p <= lim))

    m = dict(best_bid=bb, best_ask=ba, mid=mid,
             microprice=(bb * qa + ba * qb) / (qa + qb),
             spread_bps=(ba - bb) / mid * 1e4,
             depth_bid_5bps=depth(bids, 5, True), depth_ask_5bps=depth(asks, 5, False),
             depth_bid_20bps=depth(bids, 20, True), depth_ask_20bps=depth(asks, 20, False),
             coverage_bps=min(mid - bids[-1][0], asks[-1][0] - mid) / mid * 1e4)
    d = m["depth_bid_5bps"] + m["depth_ask_5bps"]
    m["imbalance_5bps"] = (m["depth_bid_5bps"] - m["depth_ask_5bps"]) / d if d else 0.0
    return m


def shock(m, venue, symbol, k):
    """Épisode de stress rejoué : l'intensité k monte de 0 à 1 puis redescend à 0."""
    if venue == "BINANCE" and symbol == "SOL-USD":   # la liquidité s'évapore sur une plateforme
        m["mid"] *= 1 - 0.006 * k                    # le prix décroche de 60 pb face aux deux autres
        m["spread_bps"] += 45 * k                    # la fourchette s'écarte de 45 pb
        for f in ("depth_bid_5bps", "depth_ask_5bps", "depth_bid_20bps", "depth_ask_20bps"):
            m[f] *= 1 - 0.85 * k                     # 85 % de la profondeur disparaît
        m["depth_bid_5bps"] *= 1 - 0.5 * k           # les acheteurs se retirent plus vite que les vendeurs
    if symbol == "USDT-USD":                         # un stablecoin perd son ancrage au dollar
        m["mid"] *= 1 - 0.0075 * k
    half = m["mid"] * m["spread_bps"] / 2e4
    m["best_bid"], m["best_ask"], m["microprice"] = m["mid"] - half, m["mid"] + half, m["mid"]
    d = m["depth_bid_5bps"] + m["depth_ask_5bps"]
    m["imbalance_5bps"] = (m["depth_bid_5bps"] - m["depth_ask_5bps"]) / d if d else 0.0


producer = EventHubProducerClient.from_connection_string(CONNECTION_STRING)
start, sent, errors = time.time(), 0, 0
ticks = int(DURATION_MIN * 60 / TICK_SEC)
with ThreadPoolExecutor(max_workers=len(FEEDS)) as pool:
    for i in range(ticks):
        t_tick = time.time()
        now = dt.datetime.now(dt.timezone.utc)
        minute = (t_tick - start) / 60
        k = 0.0
        if STRESS_TEST and STRESS_START_MIN <= minute < STRESS_START_MIN + STRESS_LEN_MIN:
            k = math.sin(math.pi * (minute - STRESS_START_MIN) / STRESS_LEN_MIN)
        futures = [pool.submit(fetch, f) for f in FEEDS]   # les 12 carnets sont lus en parallèle
        batch, n = producer.create_batch(), 0
        for (venue, symbol, klass, _), fut in zip(FEEDS, futures):
            try:
                bids, asks, latency = fut.result()
                m = liquidity(bids, asks)
            except Exception as e:                          # une plateforme qui répond mal ne bloque pas les autres
                errors += 1
                if errors <= 5 or errors % 50 == 0:
                    print(f"  {venue} {symbol} ignoré : {str(e)[:90]}")
                continue
            hit = k > 0 and ((venue == "BINANCE" and symbol == "SOL-USD") or symbol == "USDT-USD")
            if hit:
                shock(m, venue, symbol, k)
            evt = {"event_time": now.isoformat(), "venue": venue, "symbol": symbol, "asset_class": klass,
                   "best_bid": round(m["best_bid"], 6), "best_ask": round(m["best_ask"], 6),
                   "mid": round(m["mid"], 6), "microprice": round(m["microprice"], 6),
                   "spread_bps": round(m["spread_bps"], 4),
                   "depth_bid_5bps": round(m["depth_bid_5bps"], 2), "depth_ask_5bps": round(m["depth_ask_5bps"], 2),
                   "depth_bid_20bps": round(m["depth_bid_20bps"], 2), "depth_ask_20bps": round(m["depth_ask_20bps"], 2),
                   "imbalance_5bps": round(m["imbalance_5bps"], 4), "coverage_bps": round(m["coverage_bps"], 2),
                   "latency_ms": round(latency, 1), "scenario": "STRESS_TEST" if hit else "LIVE"}
            batch.add(EventData(json.dumps(evt)))
            n += 1
        if n:
            producer.send_batch(batch)
            sent += n
        if i % 12 == 0:                                     # un message par minute
            etat = f"stress {k:.0%}" if k > 0 else "marché réel"
            print(f"{now:%H:%M:%S} UTC | {etat} | événements envoyés : {fr(sent)} | erreurs : {errors}")
        time.sleep(max(0.0, TICK_SEC - (time.time() - t_tick)))
producer.close()
print(f"Terminé : {fr(sent)} événements envoyés, {errors} lectures ignorées.")